# MeteoScreening from database (influxdb) - apply
Concise, code-focused version. Explanations live in the full notebook `StepwiseMeteoScreeningFromDatabase.ipynb`.

Flow: download (`dbc-influxdb`) -> screen on high-res data (`diive`) -> resample to 30MIN -> upload.

## User settings

In [1]:
SITE = 'ch-lae'
SITE_LAT = 47.478333
SITE_LON = 8.364389

FIELDS = ['TA_T1_47_1']  # variable(s) as stored in the db ('_field')
MEASUREMENT = 'TA'      # exactly one measurement

START = '2026-03-01 00:00:01'  # start IS included
STOP = '2026-03-03 00:00:01'   # stop is NOT included

RESAMPLING_AGG = 'mean'  # 'mean' or 'sum' (use 'sum' for precipitation)

## Auto settings & imports

In [2]:
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time)
RESAMPLING_FREQ = '30min'
DIRCONF = r'F:\dev\poet\configs'

BUCKET_RAW = f'{SITE}_raw'
BUCKET_PROCESSED = f'{SITE}_processed'

In [3]:
import importlib.metadata
import warnings
from datetime import datetime

import pandas as pd
from dbc_influxdb import dbcInflux

import diive as dv

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
print(f"dbc-influxdb v{importlib.metadata.version('dbc_influxdb')}")
print(f"diive v{importlib.metadata.version('diive')}")

ModuleNotFoundError: No module named 'pytz'

## Connect & download

In [ ]:
dbc = dbcInflux(dirconf=DIRCONF)

In [ ]:
# Optional check of available fields:
# display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

In [ ]:
# Drop requested fields with no data in this period
vars_not_available = [v for v in FIELDS if v not in data_detailed.keys()]
for rem in vars_not_available:
    FIELDS.remove(rem)
    print(f'Removed {rem} (no data in period).')
print(f'Data available for: {list(data_detailed.keys())}')

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## Start MeteoScreening

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## Outlier detection
Run a test, inspect the preview, then commit it with `mscr.addflag()`. Run only the tests you need.

In [ ]:
mscr.start_outlier_detection()

Plot current cleaned data at any point during detection:

In [ ]:
for key, val in mscr.outlier_detection.items():
    val.showplot_cleaned(interactive=False)

### Manual removal

In [ ]:
REMOVE_DATES = [
    ['2024-07-14 07:15:00', '2024-07-19 00:00:15'],  # range
    # '2022-08-23 11:45:00',                          # single point
]
mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Hampel filter (day/night)

In [ ]:
mscr.flag_outliers_hampel_test(
    window_length=60 * 24 * 7,  # 7 days of 1min data
    n_sigma_daytime=5.5, n_sigma_nighttime=5.5,
    use_differencing=True, separate_daytime_nighttime=True,
    repeat=True, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Z-score (day/night)
For radiation/RH: do NOT let this remove below-zero radiation or RH>100% (both corrected later).

In [ ]:
mscr.flag_outliers_zscore_test(
    thres_zscore=4.5, separate_daytime_nighttime=True,
    lat=SITE_LAT, lon=SITE_LON, utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
    repeat=True, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Z-score rolling window

In [ ]:
mscr.flag_outliers_zscore_rolling_test(
    thres_zscore=4.5, winsize=1440 * 7, repeat=True, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Local standard deviation

In [ ]:
mscr.flag_outliers_localsd_test(
    separate_daytime_nighttime=True, n_sd=5.5, winsize=60 * 24 * 7,
    constant_sd=False, repeat=False, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Increments z-score

In [ ]:
mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Local outlier factor
Do NOT use on high-res data (1S/10S/1MIN) - very slow. Fine for half-hourly.

In [ ]:
mscr.flag_outliers_lof_test(
    n_neighbors=30, contamination=0.01, separate_daytime_nighttime=False,
    repeat=False, n_jobs=-1, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Absolute limits

In [ ]:
mscr.flag_outliers_abslim_test(minval=-18, maxval=50, showplot=True)

In [ ]:
mscr.addflag()

### Trim low

In [ ]:
mscr.flag_outliers_trim_low_test(
    trim_daytime=False, trim_nighttime=True, lower_limit=10, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Missing values flag

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall flag QCF + reports

In [ ]:
mscr.finalize_outlier_detection()

In [ ]:
mscr.report_outlier_detection_qcf_evolution()
mscr.report_outlier_detection_qcf_flags()
mscr.report_outlier_detection_qcf_series()

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## Corrections
On high-res data, after QCF. Run only what applies to the variable.

In [ ]:
mscr.showplot_cleaned()

### Remove radiation zero offset (SW_IN, SW_OUT, PPFD_IN, PPFD_OUT)

In [ ]:
mscr.correction_remove_nighttime_zero_offset()

### Remove relative humidity offset (RH)

In [ ]:
mscr.correction_remove_relativehumidity_offset()

### Set to max / min threshold

In [ ]:
mscr.correction_setto_max_threshold(threshold=30)

In [ ]:
mscr.correction_setto_min_threshold(threshold=-5)

### Set time range(s) to value

In [ ]:
DATES = [
    ['2022-04-01', '2022-04-05'],
    ['2022-09-05', '2022-09-07'],
]
mscr.correction_setto_value(dates=DATES, value=3.7, verbose=1)
mscr.showplot_cleaned(interactive=False)

### Set exact values to missing

In [ ]:
# Inspect most frequent values first:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} (top 20 of {mscr.series_hires_cleaned[ff].count()} records) ---')
    print(vc.head(20))

In [ ]:
mscr.correction_set_exact_value_to_missing(values=[0])
mscr.showplot_cleaned(interactive=False)

## Analyses (optional)
Timestamp-shift check for radiation (SW_IN, SW_OUT, PPFD_IN, PPFD_OUT).

In [ ]:
# _ = mscr.analysis_potential_radiation_correlation(utc_offset=1, mincorr=0.7, showplot=True)

## Resample to 30MIN

In [ ]:
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

In [ ]:
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else 'FAILED'
    print(f'{status} - {v}: {freq}')

## Upload to database

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload

In [ ]:
dbc = dbcInflux(dirconf=DIRCONF)
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
data_simple

In [ ]:
for v in data_detailed.keys():
    freq = dv.times.DetectFrequency(index=data_detailed[v].index, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else 'FAILED'
    print(f'{status} - {v}: {freq}')